# Uploading experiments to the local stack

This notebook talks to the REST API that `docker-compose` starts in this repository. It loads the
`Experiment` model from `model.md`, builds one record, sends it to the API, and reads the records
back.

The same `model.md` and the same database also sit behind the MCP server that Claude Desktop
connects to. A record written here is visible to the agent, and a record the agent writes is
visible here.

Start the stack before running any cell. From the `database/` folder run `./start.sh`, then check
that `http://localhost:8800/experiment/` answers and that `http://localhost:8800/docs` shows the
generated endpoints.

In [ ]:
%pip install httpx mdmodels==0.3.1 rich

In [ ]:
import rich
import httpx
import mdmodels as md

## The model

`model.md` defines four objects and one enumeration.

- `Experiment` has a name, a description, a list of `Molecule`, a list of `Enzyme`, and one
  `Conditions` block.
- `Molecule` and `Enzyme` each have a name, an initial concentration, and a `Role`.
- `Conditions` has a temperature and a pH.
- `Role` is one of `SUBSTRATE`, `PRODUCT`, `COFACTOR`, `INHIBITOR`, `ENZYME`.

`md.DataModel.from_markdown` reads that file and returns Python classes with the same names. The
generated JSON Schema for the same model is in `schemes/experiment.json`, and code for other
languages is in `code/`.

In [ ]:
library = md.DataModel.from_markdown("model.md")

print(library)

## Build one experiment

Create the `Experiment`, then attach molecules and enzymes with `add_to_molecules` and
`add_to_enzymes`. Each helper takes the fields of the nested object and appends a new entry to the
list.

In [ ]:
dataset = library.Experiment(
    name="Experiment 1",
    description="A test experiment.",
    conditions=library.Conditions(
        temperature=37,  # degrees C
        pH=7.4,
    ),
)

dataset.add_to_molecules(
    name="Molecule 1",
    initial_concentration=100,  # mM
    role=library.Role.SUBSTRATE,
)

dataset.add_to_enzymes(
    name="Enzyme 1",
    initial_concentration=100,  # ug/mL
    role=library.Role.ENZYME,
)

rich.print(dataset)

## Store it

`POST /experiment/` takes the JSON form of an `Experiment`. The API writes it across the
`experiment`, `molecule`, `enzyme`, and `conditions` tables in one transaction and returns the
stored record with the database-assigned `id`.

In [ ]:
with httpx.Client() as client:
    response = client.post(
        "http://localhost:8800/experiment/",
        json=dataset.model_dump(mode="json"),
    )
    response.raise_for_status()
    rich.print(response.json())

## Read the records back

`GET /experiment/` lists the stored experiments. Add `?full=true` to get the nested molecules,
enzymes, and conditions rather than only the top-level columns, and `?skip=` and `?limit=` to page
through the results.

For column filters, `POST /experiment/search` takes a body of the form
`{"filters": [{"table": "experiment", "filters": [{"column": "...", "operation": "equal",
"value": ...}], "connection": "and"}]}`. The operations are `equal`, `not_equal`, `greater`,
`greater_equal`, `lower`, `lower_equal`, `in`, and `not_in`. The full request schema is at
`http://localhost:8800/docs`.

In [ ]:
with httpx.Client() as client:
    response = client.get("http://localhost:8800/experiment/")
    datasets = [library.Experiment(**row) for row in response.json()]

rich.print(f"Found {len(datasets)} experiments")
for d in datasets:
    rich.print(f" - {d.name}")

In [ ]:
# The same list with the nested objects included.
with httpx.Client() as client:
    full = client.get("http://localhost:8800/experiment/", params={"full": True}).json()

rich.print(full[:1])

## The same data through the MCP server

The `mcp` service exposes the same model to Claude Desktop on port `7001`. It registers these
tools, all derived from `model.md`:

- `Get_Table_Schema` and `Get_Table_Relationships` return the columns and the foreign keys.
- `create_experiment` writes a nested `Experiment` across its tables in one transaction, the same
  operation as the `POST` above.
- `Select_Table` runs a filtered read, and `Aggregate_Table` runs a grouped count or sum.
- `Vector_Search` runs a similarity query over the `pgvector` columns.

The agent never sees raw SQL. It calls these tools, and they map onto the same tables this
notebook writes to. See the repository README for the Claude Desktop configuration and two example
conversations.